# WSR Train Detection — analysis notebook

The detection system itself lives in Python modules; this notebook is for
poking at live cameras and analysing what the watcher has captured.

| Module | Role |
| --- | --- |
| `wsr_live_capture.py` | frames from the six Railcam webcams |
| `detection_zones.py` | per-camera detect / approach / ignore polygons |
| `gala_watcher.py` | two-tier watcher: motion gate → YOLO11 episodes |
| `episode_analysis.py` | episodes ↔ scraped timetable matching |
| `classify_trains.py` | per-episode traction/livery via Gemini (structured) |


## 1. Grab a live frame and detect

YOLO11 replaces the old YOLOv5/`torch.hub` flow — same idea, current model.


In [ ]:
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO
from wsr_live_capture import CAMERAS, grab_frame

model = YOLO('yolo11s.pt')

frame = grab_frame('minehead_station')
results = model(frame, verbose=False, conf=0.35)[0]
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(results.plot(), cv2.COLOR_BGR2RGB))
plt.axis('off')
[(model.names[int(b.cls)], round(float(b.conf), 2)) for b in results.boxes]


## 2. Zone overlay

Detections only count if their box centre lands in a detect/approach zone —
this is what filters stabled stock, the Crowcombe barrow, and the Blue
Anchor camping coach.


In [ ]:
from detection_zones import draw_zones

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(draw_zones(frame, 'minehead_station'), cv2.COLOR_BGR2RGB))
plt.axis('off')


## 3. Episodes vs the timetable

Every watcher run appends to `episodes.jsonl`. Matching each episode to the
scraped timetable gives punctuality for scheduled trains — and an unmatched
episode **is** the special-train alert.


In [ ]:
import pandas as pd
from episode_analysis import match_all

matched = match_all()
if not matched:
    print('No episodes yet — run gala_watcher.py on a running day.')
else:
    df = pd.json_normalize(matched)
    cols = ['t_enter', 'camera', 'direction', 'peak_conf',
            'match.time', 'match.serviceType', 'match_gap_min', 'is_special']
    display(df[[c for c in cols if c in df.columns]])


## 4. Classify an episode

One structured Gemini call per episode, on the 1080p still. Needs a fresh
`GEMINI_API_KEY` in `.env` — the old key was exposed in git history and is burned.


In [ ]:
from episode_analysis import load_episodes
from classify_trains import classify_episode

episodes = load_episodes()
if episodes:
    print(classify_episode(episodes[-1])['classification'])
else:
    print('No episodes yet.')


## Next

- Validate direction vectors against the first known gala services
- Auto-label episode crops from timetable loco allocations
- Distil the LLM classifier into a small local model (CLIP probe / YOLO-cls)
- Feed matched events to the web app
